## Problem *(unicode1)*: Understanding Unicode (1 point)

**(a)** What Unicode character does `chr(0)` return?  
**(Ans)** Null char 

**(b)** How does this character’s string representation (`__repr__()`) differ from its printed representation?  
**(Ans)** `__repr__` -> `'\x00'`; `__str__` -> *(NULL, often invisible)*; `print` uses `__str__`.

**(c)** What happens when this character occurs in text? It may be helpful to play around with the following in your Python interpreter and see if it matches your expectations:

```python
>>> chr(0)
>>> print(chr(0))
>>> "this is a test" + chr(0) + "string"
>>> print("this is a test" + chr(0) + "string") 
``` 
**Deliverable:** Where ever print we don’t see the char but its visible when we do direct "" print as it uses `__repr__` (len captures the invisible char)



In [13]:
chr(0)

'\x00'

In [14]:
print(chr(0))

 


In [15]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [16]:
print("this is a test" + chr(0) + "string")
print(len("this is a test" + chr(0) + "string"), len("this is a test" + "string"))

this is a test string
21 20


## Problem *(unicode2)*: Unicode Encodings (3 points)

**(a)** What are some reasons to prefer training our tokenizer on **UTF-8 encoded bytes**, rather than **UTF-16** or **UTF-32**?  
It may be helpful to compare the output of these encodings for various input strings.  

**(Ans)** Utf-32, utf-16 use lot more bytes (2, 4) for encoding ascii chars whereas utf-8 uses just 1 byte, commonly used on web utf-16, utf-32 encoding have lot more zeros, most documents on net already use utf-8


**(b)** Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string into a Unicode string. Why is this function incorrect? Provide an example of an input byte string that yields incorrect results.

```python
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

>>> decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))
'hello'
```
**(Ans)** Utf-8 might be 2 or more bytes long this gives wrong result example; example try to encode and decode é



**(c)** Give a **two-byte sequence** that does **not** decode to any Unicode character(s).

**(Ans)** b"\xC0\x80"



# Problem (train_bpe_tinystories): BPE Training on TinyStories (2 points)

---

### (a)

Train a byte-level BPE tokenizer on the TinyStories dataset, using a maximum vocabulary size of **10,000**.  
Make sure to add the TinyStories `<|endoftext|>` special token to the vocabulary.  
Serialize the resulting vocabulary and merges to disk for further inspection.  

- How many hours and memory did training take?  
- What is the longest token in the vocabulary?  
- Does it make sense?  

**Resource requirements:** ≤ 30 minutes (no GPUs), ≤ 30GB RAM  

**Hint**:  
You should be able to get under 2 minutes for BPE training using `multiprocessing` during pretokenization and the following two facts:  

1. The `<|endoftext|>` token delimits documents in the data files.  
2. The `<|endoftext|>` token is handled as a special case before the BPE merges are applied.  

**Deliverable**:  
Ans) it took 294s (0.08 hrs), 6GB memory to train, the logenst token in vocabulary is ' accomplishment', yes it makes sense

---

### (b)

Profile your code. What part of the tokenizer training process takes the most time?  

**Deliverable**:  
Ans) getting the max freq pair and regex matching take most of the time in the code

# Problem (train_bpe_expts_owt): BPE Training on OpenWebText (2 points)

---

### (a)

Train a byte-level BPE tokenizer on the OpenWebText dataset, using a maximum vocabulary size of **32,000**.  
Serialize the resulting vocabulary and merges to disk for further inspection.  

- What is the longest token in the vocabulary?  
- Does it make sense?  

**Resource requirements:** ≤ 12 hours (no GPUs), ≤ 100GB RAM  

**Deliverable**:  
Ans) it took 8.33 hrs, 51GB memory to train, the logest token in vocabulary is 'ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ'

---

### (b)

Compare and contrast the tokenizer that you get training on TinyStories versus OpenWebText.  

**Deliverable**:  
A) owt has bigger tokens than tiny-stories 

In [17]:
import pstats
base_path = './artifacts/bpe/'
dataset = 'TinyStoriesV2-GPT4/'
filename = 'train-10k.prof'
stats = pstats.Stats(base_path + dataset + filename).strip_dirs().sort_stats("cumulative")
stats.print_stats(40)

Sat Sep 27 02:51:23 2025    ./artifacts/bpe/TinyStoriesV2-GPT4/train-10k.prof

         502669591 function calls (502669473 primitive calls) in 294.064 seconds

   Ordered by: cumulative time
   List reduced from 458 to 40 due to restriction <40>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    2.300    2.300  294.064  294.064 bpe.py:85(train_bpe)
     9769   85.317    0.009  173.580    0.018 {built-in method builtins.max}
        4    0.000    0.000  112.900   28.225 threading.py:611(wait)
        4    0.000    0.000  112.900   28.225 threading.py:295(wait)
       19  112.900    5.942  112.900    5.942 {method 'acquire' of '_thread.lock' objects}
        1    0.000    0.000  112.899  112.899 pool.py:369(starmap)
        1    0.000    0.000  112.898  112.898 pool.py:767(get)
        1    0.000    0.000  112.898  112.898 pool.py:764(wait)
490366278   88.263    0.000   88.263    0.000 bpe.py:126(<lambda>)
   555560    2.130    0.000    2.788    0.000 

In [18]:
import pstats
base_path = './artifacts/bpe/'
dataset = 'owt/'
filename = 'train-32k.prof'
stats = pstats.Stats(base_path + dataset + filename).strip_dirs().sort_stats("cumulative")
stats.print_stats(40)

Sat Sep 27 02:51:23 2025    ./artifacts/bpe/owt/train-32k.prof

         71237019304 function calls (71237019186 primitive calls) in 30017.329 seconds

   Ordered by: cumulative time
   List reduced from 460 to 40 due to restriction <40>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1  433.840  433.840 30017.327 30017.327 bpe.py:85(train_bpe)
    31770 15304.279    0.482 28338.675    0.892 {built-in method builtins.max}
69115890171 13034.396    0.000 13034.396    0.000 bpe.py:126(<lambda>)
 72062904  482.066    0.000  568.504    0.000 bpe.py:55(_get_pair_from_word)
        4    0.000    0.000  225.504   56.376 threading.py:611(wait)
        4    0.000    0.000  225.504   56.376 threading.py:295(wait)
       19  225.504   11.869  225.504   11.869 {method 'acquire' of '_thread.lock' objects}
        1    0.000    0.000  225.504  225.504 pool.py:369(starmap)
        1    0.000    0.000  225.503  225.503 pool.py:767(get)
        1    0.000    0.000  225.5

In [19]:
import gzip, pickle

vocab_path = 'artifacts/bpe/TinyStoriesV2-GPT4/train-10k-vocab.pkl.gz'
merges_path = 'artifacts/bpe/TinyStoriesV2-GPT4/train-10k-merges.pkl.gz'

with gzip.open(vocab_path, 'rb') as g:
    vocab = pickle.load(g)
with gzip.open(merges_path, 'rb') as g:
    merges = pickle.load(g)

print(len(vocab), len(merges))

10000 9743


In [20]:
import gzip, pickle

vocab_path = 'artifacts/bpe/owt/train-32k-vocab.pkl.gz'
merges_path = 'artifacts/bpe/owt/train-32k-merges.pkl.gz'

with gzip.open(vocab_path, 'rb') as g:
    vocab = pickle.load(g)
with gzip.open(merges_path, 'rb') as g:
    merges = pickle.load(g)

print(len(vocab), len(merges))

32000 31743


In [21]:
from cs336_basics.bpe_tokenizer import BPETokenizer
import time

import regex as re
def read_docs(path, sep="<|endoftext|>", n=10):
    docs, buf = [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if sep in line:
                left, right = line.split(sep, 1)
                buf.append(left)
                docs.append("".join(buf))
                buf = [right]
                if len(docs) >= n:
                    break
            else:
                buf.append(line)
    return docs


def compression_ratio_Bps(samples: list[str], tokenizer: "BPETokenizer") -> tuple[float, float]:
    start = time.time()
    cnt_tkn = sum(1 for _ in tokenizer.encode_iterable(samples))
    end = time.time()

    byte_cnt = sum(len(c.encode("utf-8")) for c in samples)

    ratio = byte_cnt / cnt_tkn            
    throughput = byte_cnt / (end - start) 

    return ratio, throughput

tiny_stories_samples  = read_docs('data/TinyStoriesV2-GPT4-train.txt')
tiny_tokenizer = BPETokenizer.from_files('./artifacts/bpe/TinyStoriesV2-GPT4/train-10k-vocab.pkl.gz', './artifacts/bpe/TinyStoriesV2-GPT4/train-10k-merges.pkl.gz')
print(max(tiny_tokenizer.vocab.values(), key=len), len(tiny_tokenizer.merges))
print(tiny_stories_samples)
print(compression_ratio_Bps(tiny_stories_samples, tiny_tokenizer))

b' accomplishment' 9743
["\nOnce upon a time there was a little boy named Ben. Ben loved to explore the world around him. He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was walking through the store when he came across a very special vase. When Ben saw it he was amazed!  \nHe said, “Wow, that is a really amazing vase! Can I buy it?” \nThe shopkeeper smiled and said, “Of course you can. You can take it home and show all your friends how amazing it is!”\nSo Ben took the vase home and he was so proud of it! He called his friends over and showed them the amazing vase. All his friends thought the vase was beautiful and couldn't believe how lucky Ben was. \nAnd that's how Ben found an amazing vase in the store!\n", '\nOnce upon a time, there was a reliable otter named Ollie. He lived in a river with his family. They all loved to play and swim together.\nOne day, Ollie\'s mom said, "Ollie, hurry and get some fish for dinner!" Ollie swam fast to 

In [22]:
owt_samples = read_docs('data/owt-train.txt')
owt_tokenizer = BPETokenizer.from_files('./artifacts/bpe/owt/train-32k-vocab.pkl.gz', './artifacts/bpe/owt/train-32k-merges.pkl.gz')
print(compression_ratio_Bps(owt_samples, owt_tokenizer))
print(compression_ratio_Bps(owt_samples, tiny_tokenizer))
hours_req = ((825 * 1000)/1.6)/(60*60)
print(f'{hours_req:.2f} hours for 825GB of data' )

(4.691150178784267, 574222.6871834116)
(3.1892028765319558, 705941.1050364019)
143.23 hours for 825GB of data


In [23]:
# import numpy as np

# with open("data/owt-train.txt", "r", encoding="utf-8") as f:
#     text = f.read()

# tokens = np.array(owt_tokenizer.encode(text), dtype=np.int32)

# memmap = np.memmap("owt_tokens.memmap", dtype=np.int32, mode="w+", shape=tokens.shape)
# memmap[:] = tokens
# memmap.flush()


In [24]:
import torch
print(torch.__version__)

a = torch.randn(2,3, 4).T
print(a.shape)

2.6.0+cu124
torch.Size([4, 3, 2])


/tmp/ipykernel_260829/2108997560.py:4: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3725.)
  a = torch.randn(2,3, 4).T


In [25]:
import numpy as np
tokens = np.memmap("artifacts/bpe/TinyStoriesV2-GPT4/TinyStoriesV2-GPT4-train.memmap", dtype=np.int32, mode="r")
print(len(tokens))
print(tiny_tokenizer.decode(tokens[:100]))

560259947

Once upon a time there was a little boy named Ben. Ben loved to explore the world around him. He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was walking through the store when he came across a very special vase. When Ben saw it he was amazed!  
He said, “Wow, that is a really amazing vase! Can I buy it?” 
The shopkeeper smiled and said, “Of course you can.


In [26]:
a = tiny_tokenizer.encode("Long ago, in a tiny village")

In [27]:
a = torch.tensor(a).long()
a.repeat(2, 1).shape

torch.Size([2, 8])

In [28]:
a = torch.tensor([1, 3, 5])
a > 3

tensor([False, False,  True])

In [41]:
from cs336_basics.layers import MyTransformer, MyConfig
from cs336_basics.utils import my_load_checkpoint, generate_seq
import torch.optim as optim

max_seq_len = 256
config = MyConfig(
    num_layers=4,
    d_model=512,
    num_heads=16,
    d_ff=1344,
    max_seq_len=max_seq_len,
    rope_theta=10000.0,
    vocab_size=10000,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = MyTransformer(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.5)


tiny_tokenizer = BPETokenizer.from_files(
    './artifacts/bpe/TinyStoriesV2-GPT4/train-10k-vocab.pkl.gz',
    './artifacts/bpe/TinyStoriesV2-GPT4/train-10k-merges.pkl.gz'
)

my_load_checkpoint('./artifacts/checkpoints/checkpoint_33000', model, optimizer)

start_seq = torch.tensor(tiny_tokenizer.encode("Long ago, there was "))
num_samples = 4
out_seq = generate_seq(model, start_seq, device=device)
print('-----------------')
samples = []
for i in range(num_samples):
    tokens = out_seq[i, :].tolist()
    decoded = tiny_tokenizer.decode(tokens)
    print(decoded)
    print('-----------------')
    samples.append([decoded])



-----------------
Long ago, there was ! We wanted to make a bridge over this cold pond over there to help us wash the water off. Holly was happy that she was able to help the bridge and made it safely! 
Andy was proud of Martha for being so hurry and helping the bridge. Before the sun rose near the icy pond, the train was fixed and connected Andy to a warm and dry pond. 
Metey happily went on its way. She saw lots of delicious camel
-----------------
Long ago, there was idential and hundreds of other children eating things. Pat was so excited to receive the rough lunch from Happy and took a big open napkin tree to show his family. 
When Pat returned, he thanked the children for the lunch and ran off to play. He waved goodbye to the children and went home with his treats. Pat never forgot the lesson he had served with food and most of his wildness.
<|endoftext|>

-----------------
Long ago, there was ...jinkle, a small and modest bunny!
Maddie was so excited and couldn't wait to explore